# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


In [ ]:
import os
from pathlib import Path
import csv
import sys

import numpy as np
import matplotlib.pyplot as plt

from pydrake.all import QuasidynamicLinearPusher

PROJECT_DIR = Path(os.environ.get("PUSHER_PROJECT", Path.cwd().parent)).expanduser().resolve()
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Known model parameters.
K_A = 1.0
H = 0.02
W = 0.1

# Every identification trial starts here.
INITIAL_STATE = np.array([0.0, 0.2])
MASS_PRIOR = np.array([0.005, 0.030])

TEST_MASSES = np.array([
    0.005,
    0.010,
    0.015,
    0.0175,
    0.020,
    0.025,
    0.030,
])
TEST_SEEDS = list(range(10))

PROBE_EXCITATIONS = np.array([
    0.005, 0.010, 0.015, 0.020, 0.010, 0.020
])

# Noise is added to displacement only.
DISPLACEMENT_NOISE_BOUND = 10e-6

# Empirical numerical allowance, not a global certified bound.
NUMERICAL_ALLOWANCE = 5e-6
RESIDUAL_BOUND = (
    DISPLACEMENT_NOISE_BOUND + NUMERICAL_ALLOWANCE
)

TASK_DISTANCE = 0.005
TASK_STEPS = 40
MAX_PLANNED_EXCITATION = 0.020

METHODS = ["Before learning", "After learning", "Oracle"]

print("Python:", sys.executable)
print("Total trials:", len(TEST_MASSES) * len(TEST_SEEDS))
print("Task duration [s]:", TASK_STEPS * H)
print("Residual allowance [um]:", RESIDUAL_BOUND * 1e6)
print("Results directory:", RESULTS_DIR)

In [ ]:
class DrakePusher:
    """Evaluate one step of the unsmoothed Drake linear pusher."""

    def __init__(self, mass):
        self.system = QuasidynamicLinearPusher(
            k_a=K_A,
            m_o=float(mass),
            w=W,
            dt=H,
            dynamics_smoothing=0.0,
        )

    def step(self, state, command):
        context = self.system.CreateDefaultContext()

        context.SetDiscreteState(
            np.asarray(state, dtype=float)
        )
        self.system.get_input_port().FixValue(
            context, np.array([float(command)])
        )

        update = self.system.EvalUniquePeriodicDiscreteUpdate(
            context
        )
        context.SetDiscreteState(update)

        next_state = (
            context.get_discrete_state_vector().CopyToVector()
        )

        if not np.all(np.isfinite(next_state)):
            raise RuntimeError("Drake returned a nonfinite state.")

        return next_state


def alpha_from_mass(mass):
    return K_A / (K_A + np.asarray(mass) / H**2)


def mass_from_alpha(alpha):
    return K_A * H**2 * (1.0 / np.asarray(alpha) - 1.0)


def analytical_step(state, command, mass):
    object_position = float(state[1])

    excitation = max(
        0.0, float(command) + W - object_position
    )

    alpha = float(alpha_from_mass(mass))
    next_object = object_position + alpha * excitation

    if excitation > 0.0:
        next_pusher = next_object - W
    else:
        next_pusher = float(command)

    return np.array([next_pusher, next_object])


print("Simulator and analytical model ready.")

In [ ]:
class MassIntervalEstimator:
    def __init__(self, mass_prior, residual_bound):
        m_low, m_high = np.asarray(mass_prior, dtype=float)

        if not (0 < m_low <= m_high):
            raise ValueError("Invalid mass prior.")

        self.alpha_low = float(alpha_from_mass(m_high))
        self.alpha_high = float(alpha_from_mass(m_low))
        self.residual_bound = float(residual_bound)

    def update(self, excitation, measured_displacement):
        a = float(excitation)
        d = float(measured_displacement)

        if a <= 0:
            raise ValueError("Positive excitation is required.")

        observation_low = (
            d - self.residual_bound
        ) / a
        observation_high = (
            d + self.residual_bound
        ) / a

        new_low = max(self.alpha_low, observation_low)
        new_high = min(self.alpha_high, observation_high)

        if new_low > new_high:
            raise RuntimeError("Parameter interval became empty.")

        self.alpha_low = new_low
        self.alpha_high = new_high

    def mass_bounds(self):
        return np.array([
            mass_from_alpha(self.alpha_high),
            mass_from_alpha(self.alpha_low),
        ], dtype=float)

    def estimate(self):
        return float(np.mean(self.mass_bounds()))


print("Estimator ready.")

In [ ]:
def identify_mass_for_trial(true_mass, seed):
    rng = np.random.default_rng(seed)
    plant = DrakePusher(true_mass)
    estimator = MassIntervalEstimator(
        MASS_PRIOR, RESIDUAL_BOUND
    )

    state = INITIAL_STATE.copy()
    mass_history = [estimator.mass_bounds()]
    numerical_residuals = []
    probe_log = []

    coverage_tolerance = 1e-12  # kg; floating-point roundoff only

    for step, requested_excitation in enumerate(
        PROBE_EXCITATIONS, start=1
    ):
        command = state[1] - W + requested_excitation
        excitation = max(
            0.0, command + W - state[1]
        )

        next_state = plant.step(state, command)

        # Evaluation only: not passed to the estimator.
        analytical_true = analytical_step(
            state, command, true_mass
        )
        numerical_residual = (
            next_state[1] - analytical_true[1]
        )
        numerical_residuals.append(numerical_residual)

        if abs(numerical_residual) > NUMERICAL_ALLOWANCE:
            raise RuntimeError(
                "Identification numerical discrepancy "
                "exceeded the allowance."
            )

        actual_displacement = next_state[1] - state[1]
        measurement_noise = rng.uniform(
            -DISPLACEMENT_NOISE_BOUND,
            DISPLACEMENT_NOISE_BOUND,
        )
        measured_displacement = (
            actual_displacement + measurement_noise
        )

        estimator.update(
            excitation, measured_displacement
        )
        bounds = estimator.mass_bounds()
        mass_history.append(bounds)

        probe_log.append({
            "step": step,
            "command": command,
            "excitation": excitation,
            "measured_displacement": measured_displacement,
            "mass_low": bounds[0],
            "mass_high": bounds[1],
        })

        state = next_state

    mass_history = np.asarray(mass_history)

    covered_each_step = (
        (mass_history[:, 0] - coverage_tolerance <= true_mass)
        & (true_mass <= mass_history[:, 1] + coverage_tolerance)
    )

    return {
        "estimate": estimator.estimate(),
        "bounds": estimator.mass_bounds(),
        "covered": bool(np.all(covered_each_step)),
        "post_probe_state": state.copy(),
        "mass_history": mass_history,
        "probe_log": probe_log,
        "max_numerical_error": float(
            np.max(np.abs(numerical_residuals))
        ),
    }


print("Identification trial function ready.")

In [ ]:
def make_open_loop_plan(initial_state, goal, mass, num_steps):
    initial_state = np.asarray(
        initial_state, dtype=float
    ).copy()

    desired_increment = (
        goal - initial_state[1]
    ) / num_steps

    if desired_increment <= 0:
        raise ValueError("The task must be a forward push.")

    alpha = float(alpha_from_mass(mass))
    excitation = desired_increment / alpha

    if excitation > MAX_PLANNED_EXCITATION:
        raise ValueError(
            f"Required excitation {excitation:.6f} m "
            "exceeds the planned limit."
        )

    nominal_states = [initial_state]
    controls = []

    for _ in range(num_steps):
        nominal_state = nominal_states[-1]

        command = (
            nominal_state[1] - W + excitation
        )

        next_nominal = analytical_step(
            nominal_state, command, mass
        )

        controls.append(command)
        nominal_states.append(next_nominal)

    nominal_states = np.asarray(nominal_states)

    if not np.isclose(
        nominal_states[-1, 1],
        goal,
        atol=1e-10,
        rtol=0,
    ):
        raise RuntimeError("Nominal plan missed its endpoint.")

    return {
        "nominal_states": nominal_states,
        "controls": np.asarray(controls),
        "planned_excitation": excitation,
    }


def execute_open_loop(initial_state, controls, true_mass):
    plant = DrakePusher(true_mass)

    states = [
        np.asarray(initial_state, dtype=float).copy()
    ]
    excitations = []
    numerical_residuals = []

    for command in controls:
        current_state = states[-1]

        excitation = max(
            0.0,
            float(command) + W - current_state[1],
        )

        next_state = plant.step(current_state, command)

        # Evaluation of one-step numerical discrepancy.
        analytical_true = analytical_step(
            current_state, command, true_mass
        )
        numerical_residuals.append(
            next_state[1] - analytical_true[1]
        )

        excitations.append(excitation)
        states.append(next_state)

    return {
        "states": np.asarray(states),
        "excitations": np.asarray(excitations),
        "max_numerical_error": float(
            np.max(np.abs(numerical_residuals))
        ),
    }


print("Planning and execution functions ready.")

In [ ]:
batch_rows = []
batch_failures = []
trial_details = []

total_trials = len(TEST_MASSES) * len(TEST_SEEDS)
trial_number = 0

for true_mass in TEST_MASSES:
    for seed in TEST_SEEDS:
        trial_number += 1
        phase = "identification"

        try:
            identification = identify_mass_for_trial(
                float(true_mass), seed
            )

            start = identification["post_probe_state"]
            goal = start[1] + TASK_DISTANCE

            planning_masses = {
                "Before learning": float(np.mean(MASS_PRIOR)),
                "After learning": identification["estimate"],
                "Oracle": float(true_mass),
            }

            row = {
                "true_mass": float(true_mass),
                "seed": seed,
                "estimated_mass": identification["estimate"],
                "mass_low": identification["bounds"][0],
                "mass_high": identification["bounds"][1],
                "interval_width": (
                    identification["bounds"][1]
                    - identification["bounds"][0]
                ),
                "covered": identification["covered"],
                "identification_max_numerical_error": (
                    identification["max_numerical_error"]
                ),
            }

            method_details = {}

            for name, planning_mass in planning_masses.items():
                phase = f"{name}: planning"

                plan = make_open_loop_plan(
                    start,
                    goal,
                    planning_mass,
                    TASK_STEPS,
                )

                phase = f"{name}: execution"

                execution = execute_open_loop(
                    start,
                    plan["controls"],
                    float(true_mass),
                )

                actual = execution["states"]
                tracking_error = (
                    actual[:, 1]
                    - plan["nominal_states"][:, 1]
                )

                row[name] = float(actual[-1, 1] - goal)
                row[f"{name} rmse"] = float(
                    np.sqrt(np.mean(tracking_error[1:]**2))
                )
                row[f"{name} min excitation"] = float(
                    execution["excitations"].min()
                )
                row[f"{name} max excitation"] = float(
                    execution["excitations"].max()
                )
                row[f"{name} max numerical error"] = (
                    execution["max_numerical_error"]
                )

                method_details[name] = {
                    "plan": plan,
                    "execution": execution,
                }

            batch_rows.append(row)
            trial_details.append({
                "true_mass": float(true_mass),
                "seed": seed,
                "identification": identification,
                "methods": method_details,
            })

        except RuntimeError as error:
            batch_failures.append({
                "true_mass": float(true_mass),
                "seed": seed,
                "phase": phase,
                "reason": str(error),
            })

        print(
            f"\rProcessed {trial_number}/{total_trials} trials",
            end="",
            flush=True,
        )

print()
print("Completed trials:", len(batch_rows))
print("Runtime-failed trials:", len(batch_failures))

for failure in batch_failures:
    print(failure)

In [ ]:
if not batch_rows:
    raise RuntimeError(
        "No completed trials. Inspect the failure messages."
    )

print(
    f"{'Mass [g]':>9}"
    f"{'Trials':>8}"
    f"{'Coverage':>11}"
    f"{'Before MAE [um]':>18}"
    f"{'After MAE [um]':>18}"
    f"{'Oracle MAE [um]':>18}"
    f"{'Width [g]':>12}"
)

for mass in TEST_MASSES:
    selected = [
        row for row in batch_rows
        if np.isclose(
            row["true_mass"], mass, atol=1e-12, rtol=0
        )
    ]

    if not selected:
        print(f"{1000 * mass:>9.1f}  No completed trials")
        continue

    coverage = np.mean([
        row["covered"] for row in selected
    ])

    mean_errors = {
        name: np.mean([
            abs(row[name]) for row in selected
        ])
        for name in METHODS
    }

    width = np.mean([
        row["interval_width"] for row in selected
    ])

    print(
        f"{1000 * mass:>9.1f}"
        f"{len(selected):>8}"
        f"{100 * coverage:>10.1f}%"
        f"{1e6 * mean_errors['Before learning']:>18.3f}"
        f"{1e6 * mean_errors['After learning']:>18.3f}"
        f"{1e6 * mean_errors['Oracle']:>18.3f}"
        f"{1000 * width:>12.4f}"
    )

coverage_failures = [
    row for row in batch_rows if not row["covered"]
]

improvement_count = sum(
    abs(row["After learning"]) < abs(row["Before learning"])
    for row in batch_rows
)

print("\nMAE = mean absolute final task error across seeds.")
print("Coverage = truth inside every identification interval.")
print("Statistics describe completed trials.")

print("\nRuntime failures:", len(batch_failures))
print("Coverage failures:", len(coverage_failures))
print(
    "Trials with smaller endpoint error after learning:",
    f"{improvement_count}/{len(batch_rows)}",
)

print(
    "Largest identification numerical discrepancy [um]:",
    1e6 * max(
        row["identification_max_numerical_error"]
        for row in batch_rows
    ),
)

for name in METHODS:
    max_error = max(
        row[f"{name} max numerical error"]
        for row in batch_rows
    )
    print(
        f"Largest task numerical discrepancy, {name} [um]: "
        f"{1e6 * max_error:.6f}"
    )

if coverage_failures:
    print("\nTrials with coverage failure:")
    for row in coverage_failures:
        print(
            "mass [g] =", 1000 * row["true_mass"],
            "seed =", row["seed"],
        )

In [ ]:
available_masses = [
    mass for mass in TEST_MASSES
    if any(
        np.isclose(
            row["true_mass"], mass, atol=1e-12, rtol=0
        )
        for row in batch_rows
    )
]

mass_axis = 1000 * np.asarray(available_masses)

batch_figure, axes = plt.subplots(
    1, 3, figsize=(16, 4)
)

for name in METHODS:
    mean_errors = []

    for mass in available_masses:
        selected = [
            row for row in batch_rows
            if np.isclose(
                row["true_mass"], mass, atol=1e-12, rtol=0
            )
        ]

        mean_errors.append(
            1e6 * np.mean([
                abs(row[name]) for row in selected
            ])
        )

    axes[0].plot(
        mass_axis, mean_errors, "o-", label=name
    )

axes[0].set_xlabel("True mass [g]")
axes[0].set_ylabel("Mean absolute endpoint error [um]")
axes[0].set_title("Task performance")
axes[0].legend()

for mass in available_masses:
    selected = [
        row for row in batch_rows
        if np.isclose(
            row["true_mass"], mass, atol=1e-12, rtol=0
        )
    ]

    estimates = 1000 * np.array([
        row["estimated_mass"] for row in selected
    ])

    axes[1].scatter(
        np.full(len(estimates), 1000 * mass),
        estimates,
        alpha=0.6,
        color="tab:blue",
    )

axes[1].plot(
    mass_axis, mass_axis, "k--", label="Exact estimate"
)
axes[1].set_xlabel("True mass [g]")
axes[1].set_ylabel("Estimated mass [g]")
axes[1].set_title("Each point is one noise seed")
axes[1].legend()

learned_errors = [
    [
        1e6 * abs(row["After learning"])
        for row in batch_rows
        if np.isclose(
            row["true_mass"], mass, atol=1e-12, rtol=0
        )
    ]
    for mass in available_masses
]

axes[2].boxplot(learned_errors)
axes[2].set_xticks(
    np.arange(1, len(available_masses) + 1)
)
axes[2].set_xticklabels([
    f"{1000 * mass:g}" for mass in available_masses
])
axes[2].set_xlabel("True mass [g]")
axes[2].set_ylabel("Absolute endpoint error [um]")
axes[2].set_title("Learned-model variation across seeds")

for ax in axes:
    ax.grid(True, alpha=0.3)

batch_figure.tight_layout()
plt.show()

In [ ]:
uncertainty_figure, ax = plt.subplots(figsize=(8, 4))

for mass in available_masses:
    selected = [
        trial for trial in trial_details
        if np.isclose(
            trial["true_mass"], mass, atol=1e-12, rtol=0
        )
    ]

    widths = np.stack([
        1000 * (
            trial["identification"]["mass_history"][:, 1]
            - trial["identification"]["mass_history"][:, 0]
        )
        for trial in selected
    ])

    steps = np.arange(widths.shape[1])

    ax.plot(
        steps,
        widths.mean(axis=0),
        "o-",
        label=f"{1000 * mass:g} g",
    )

ax.set_xlabel("Number of probes")
ax.set_ylabel("Mean mass-interval width [g]")
ax.set_title("Uncertainty reduction across hidden masses")
ax.grid(True, alpha=0.3)
ax.legend(title="True mass")

uncertainty_figure.tight_layout()
plt.show()

In [ ]:
run_number = 1

while True:
    output_dir = (
        RESULTS_DIR / f"batch_validation_{run_number:03d}"
    )

    try:
        output_dir.mkdir(parents=True, exist_ok=False)
        break
    except FileExistsError:
        run_number += 1

if batch_rows:
    with (output_dir / "completed_trials.csv").open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file, fieldnames=list(batch_rows[0].keys())
        )
        writer.writeheader()
        writer.writerows(batch_rows)

with (output_dir / "failed_trials.csv").open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["true_mass", "seed", "phase", "reason"],
    )
    writer.writeheader()
    writer.writerows(batch_failures)

np.savez(
    output_dir / "settings.npz",
    test_masses=TEST_MASSES,
    test_seeds=np.asarray(TEST_SEEDS),
    initial_state=INITIAL_STATE,
    mass_prior=MASS_PRIOR,
    probe_excitations=PROBE_EXCITATIONS,
    displacement_noise_bound=DISPLACEMENT_NOISE_BOUND,
    numerical_allowance=NUMERICAL_ALLOWANCE,
    task_distance=TASK_DISTANCE,
    task_steps=TASK_STEPS,
    max_planned_excitation=MAX_PLANNED_EXCITATION,
    k_a=K_A,
    h=H,
    w=W,
)

if trial_details:
    np.savez(
        output_dir / "trial_arrays.npz",
        true_masses=np.array([
            trial["true_mass"] for trial in trial_details
        ]),
        seeds=np.array([
            trial["seed"] for trial in trial_details
        ]),
        method_names=np.asarray(METHODS),
        mass_histories=np.stack([
            trial["identification"]["mass_history"]
            for trial in trial_details
        ]),
        post_probe_states=np.stack([
            trial["identification"]["post_probe_state"]
            for trial in trial_details
        ]),
        controls=np.stack([
            np.stack([
                trial["methods"][name]["plan"]["controls"]
                for name in METHODS
            ])
            for trial in trial_details
        ]),
        nominal_states=np.stack([
            np.stack([
                trial["methods"][name]["plan"]["nominal_states"]
                for name in METHODS
            ])
            for trial in trial_details
        ]),
        actual_states=np.stack([
            np.stack([
                trial["methods"][name]["execution"]["states"]
                for name in METHODS
            ])
            for trial in trial_details
        ]),
    )

batch_figure.savefig(
    output_dir / "batch_comparison.png",
    dpi=200,
    bbox_inches="tight",
)
uncertainty_figure.savefig(
    output_dir / "uncertainty_reduction.png",
    dpi=200,
    bbox_inches="tight",
)

print("Saved results to:", output_dir)